# Module 1 Homework — 2026 Cohort
### Stock Market Analytics Zoomcamp — Introduction and Data Sources

هذا الـ notebook بيحل الأسئلة الأربعة الأساسية في `cohorts/2026/homework1.md`:
1. S&P 500 — أكتر سنة (من 2020) دخلها شركات جديدة للمؤشر
2. Macro — مقارنة عائد YTD للمؤشرات العالمية مع S&P 500 (حتى 21 أغسطس 2026)
3. S&P 500 — تحليل الـ market corrections (median drawdown)
4. Amazon (AMZN) — تحليل رد فعل السهم بعد مفاجآت الأرباح (earnings surprises)

بالإضافة لأسئلة 5 و6 (النصية/الاستكشافية) كتمبليت جاهز تكمله بنفسك.

## 0) Imports

In [ ]:
import numpy as np
import pandas as pd
import requests
import yfinance as yf
import matplotlib.pyplot as plt

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                  '(KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}


## Question 1 — S&P 500 Stocks Added to the Index

**السؤال:** أي سنة (من 2020) شافت أعلى عدد إضافات لشركات جديدة في مؤشر S&P 500؟

In [ ]:
url_sp500 = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
response = requests.get(url_sp500, headers=headers, timeout=30)
response.raise_for_status()

tables = pd.read_html(response.text)

# أول جدول عادة هو قائمة الشركات الحالية فيها عمود 'Date added'
sp500_df = tables[0].copy()
print(sp500_df.columns.tolist())
sp500_df.head(3)


In [ ]:
# تنضيف عمود التاريخ
sp500_df['Date added'] = pd.to_datetime(sp500_df['Date added'], errors='coerce')
sp500_df['Year_added'] = sp500_df['Date added'].dt.year

# عدد الإضافات لكل سنة بداية من 2020
additions_from_2020 = sp500_df[sp500_df['Year_added'] >= 2020]
additions_by_year = additions_from_2020.groupby('Year_added').size().sort_values(ascending=False)

print("عدد الإضافات لكل سنة (من 2020):")
print(additions_by_year)

best_year = int(additions_by_year.index[0])
print(f"\n>>> أعلى سنة إضافات (من 2020) = {best_year} بعدد {int(additions_by_year.iloc[0])} إضافة")


In [ ]:
# رسم بياني لعدد الإضافات حسب السنة
fig, ax = plt.subplots(figsize=(8, 4.5))
additions_by_year.sort_index().plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_title('S&P 500: Number of New Additions per Year (2020+)')
ax.set_xlabel('Year')
ax.set_ylabel('Number of additions')
plt.tight_layout()
plt.show()


### Additional: كام سهم في المؤشر حاليًا موجود من أكتر من 20 سنة؟

In [ ]:
today = pd.Timestamp.today()
cutoff_20y = today - pd.DateOffset(years=20)

stocks_over_20y = sp500_df[sp500_df['Date added'] <= cutoff_20y]
print(f"عدد الأسهم اللي دخلت المؤشر من أكتر من 20 سنة (قبل {cutoff_20y.date()}): {len(stocks_over_20y)}")
print(f"من إجمالي {sp500_df['Date added'].notna().sum()} سهم عندهم تاريخ إضافة معروف")


## Question 2 — Indexes YTD Performance (as of 21 August 2026)

**السؤال:** كام مؤشر (من أصل 10) عنده عائد YTD أفضل من S&P 500 حتى 21 أغسطس 2026؟

In [ ]:
indices = {
    'United States (S&P 500)': '^GSPC',
    'China (Shanghai Composite)': '000001.SS',
    'Hong Kong (HANG SENG)': '^HSI',
    'Australia (S&P/ASX 200)': '^AXJO',
    'India (Nifty 50)': '^NSEI',
    'Canada (S&P/TSX Composite)': '^GSPTSE',
    'Germany (DAX)': '^GDAXI',
    'United Kingdom (FTSE 100)': '^FTSE',
    'Japan (Nikkei 225)': '^N225',
    'Mexico (IPC Mexico)': '^MXX',
    'Brazil (Ibovespa)': '^BVSP',
}

start_date = '2026-01-01'
end_date = '2026-08-21'

ytd_returns = {}
for name, ticker in indices.items():
    try:
        df = yf.download(ticker, start=start_date, end=end_date, progress=False)
        if df.empty:
            print(f"[WARN] No data for {name} ({ticker})")
            continue
        df = df.sort_index()
        close_col = 'Close' if 'Close' in df.columns else df.columns[0]
        first_close = df[close_col].iloc[0]
        last_close = df[close_col].iloc[-1]
        ytd_return = (last_close / first_close) - 1
        ytd_returns[name] = ytd_return
    except Exception as e:
        print(f"[WARN] Failed to download {name} ({ticker}): {e}")

ytd_df = pd.Series(ytd_returns, name='YTD_return').sort_values(ascending=False).to_frame()
ytd_df['YTD_return_%'] = (ytd_df['YTD_return'] * 100).round(2)
ytd_df


In [ ]:
us_ytd = ytd_returns.get('United States (S&P 500)', np.nan)
better_than_us = {k: v for k, v in ytd_returns.items() if k != 'United States (S&P 500)' and v > us_ytd}

print(f"عائد S&P 500 YTD = {us_ytd:.2%}")
print(f"\nعدد المؤشرات (من أصل 10 غير أمريكا) اللي أداؤها أفضل من S&P 500 = {len(better_than_us)}")
for k, v in sorted(better_than_us.items(), key=lambda x: -x[1]):
    print(f"  - {k}: {v:.2%}")


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
colors = ['crimson' if k == 'United States (S&P 500)' else 'steelblue' for k in ytd_df.index]
ax.barh(ytd_df.index, ytd_df['YTD_return_%'], color=colors, edgecolor='black')
ax.axvline(us_ytd * 100, color='crimson', linestyle='--', alpha=0.6, label='S&P 500 YTD')
ax.set_title('YTD Returns by Country Index (as of 21 Aug 2026)')
ax.set_xlabel('YTD Return (%)')
ax.legend()
plt.tight_layout()
plt.show()


### Additional: أداء المؤشرات على 3، 5، 10 سنين مقارنة بـ S&P 500

In [ ]:
def period_return(ticker, years, end_date='2026-08-21'):
    end_dt = pd.Timestamp(end_date)
    start_dt = end_dt - pd.DateOffset(years=years)
    df = yf.download(ticker, start=start_dt.strftime('%Y-%m-%d'), end=end_date, progress=False)
    if df.empty:
        return np.nan
    df = df.sort_index()
    close_col = 'Close' if 'Close' in df.columns else df.columns[0]
    return df[close_col].iloc[-1] / df[close_col].iloc[0] - 1

multi_period_returns = {}
for name, ticker in indices.items():
    row = {}
    for yrs in [3, 5, 10]:
        try:
            row[f'{yrs}Y_return'] = period_return(ticker, yrs)
        except Exception as e:
            print(f"[WARN] {name} {yrs}Y failed: {e}")
            row[f'{yrs}Y_return'] = np.nan
    multi_period_returns[name] = row

multi_period_df = pd.DataFrame(multi_period_returns).T
us_row = multi_period_df.loc['United States (S&P 500)']

for yrs in [3, 5, 10]:
    col = f'{yrs}Y_return'
    n_better = (multi_period_df[col] > us_row[col]).sum() - (1 if False else 0)
    # نطرح صف أمريكا نفسه من العد
    n_better = (multi_period_df.drop('United States (S&P 500)')[col] > us_row[col]).sum()
    print(f"{yrs}-year: {n_better} مؤشر أداؤه أفضل من S&P 500 (S&P 500 = {us_row[col]:.1%})")

multi_period_df.style.format('{:.1%}')


## Question 3 — S&P 500 Market Corrections Analysis

**السؤال:** احسب الـ median drawdown (%) لتصحيحات السوق الملحوظة (≥5% هبوط من أعلى قمة سابقة) في S&P 500 من 1950 لحد دلوقتي.

In [ ]:
sp500_hist = yf.download('^GSPC', start='1950-01-01', progress=False)
sp500_hist = sp500_hist.sort_index()
close_col = 'Close' if 'Close' in sp500_hist.columns else sp500_hist.columns[0]
close = sp500_hist[close_col].dropna()

print(f"Data range: {close.index.min().date()} to {close.index.max().date()}  ({len(close)} trading days)")


In [ ]:
# 1) نحدد كل الأيام اللي فيها the price بيعمل all-time high جديد (>= كل القيم اللي قبلها)
running_max = close.cummax()
is_ath = close >= running_max
ath_dates = close.index[is_ath]
ath_prices = close[is_ath]

print(f"Number of all-time-high days identified: {len(ath_dates)}")

# 2) لكل زوج قمم متتاليين، نلاقي أقل سعر بينهم، ونحسب الـ drawdown والمدة
corrections = []
for i in range(len(ath_dates) - 1):
    peak_date = ath_dates[i]
    next_peak_date = ath_dates[i + 1]
    peak_price = ath_prices.iloc[i]

    segment = close.loc[peak_date:next_peak_date]
    trough_price = segment.min()
    trough_date = segment.idxmin()

    drawdown_pct = (peak_price - trough_price) / peak_price * 100

    if drawdown_pct >= 5:  # نفلتر بس الـ corrections الملحوظة (>= 5%)
        duration_days = (trough_date - peak_date).days
        corrections.append({
            'peak_date': peak_date, 'peak_price': float(peak_price),
            'trough_date': trough_date, 'trough_price': float(trough_price),
            'drawdown_pct': drawdown_pct, 'duration_days': duration_days,
        })

corrections_df = pd.DataFrame(corrections)
print(f"\nNumber of corrections with drawdown >= 5%: {len(corrections_df)}")
corrections_df.sort_values('drawdown_pct', ascending=False).head(10)


In [ ]:
# مقارنة مع أكبر 10 تصحيحات معروفة تاريخيًا (من نص الواجب) كـ sanity check
print("Top 10 largest corrections found in our data:")
display(corrections_df.sort_values('drawdown_pct', ascending=False).head(10)
        [['peak_date', 'trough_date', 'drawdown_pct', 'duration_days']])


In [ ]:
quantiles = [0.25, 0.5, 0.75]

drawdown_quantiles = corrections_df['drawdown_pct'].quantile(quantiles)
duration_quantiles = corrections_df['duration_days'].quantile(quantiles)

summary_q3 = pd.DataFrame({
    'drawdown_pct': drawdown_quantiles,
    'duration_days': duration_quantiles,
})
summary_q3.index = ['25th percentile', '50th percentile (median)', '75th percentile']
display(summary_q3)

median_drawdown = corrections_df['drawdown_pct'].median()
median_duration = corrections_df['duration_days'].median()
print(f"\n>>> Median drawdown of significant corrections = {median_drawdown:.1f}%")
print(f">>> Median duration = {median_duration:.0f} days")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(corrections_df['drawdown_pct'], bins=20, color='indianred', edgecolor='black')
axes[0].axvline(median_drawdown, color='black', linestyle='--', label=f'median = {median_drawdown:.1f}%')
axes[0].set_title('Distribution of Correction Drawdowns')
axes[0].set_xlabel('Drawdown (%)')
axes[0].legend()

axes[1].hist(corrections_df['duration_days'], bins=20, color='seagreen', edgecolor='black')
axes[1].axvline(median_duration, color='black', linestyle='--', label=f'median = {median_duration:.0f}d')
axes[1].set_title('Distribution of Correction Durations')
axes[1].set_xlabel('Duration (days)')
axes[1].legend()

plt.tight_layout()
plt.show()


## Question 4 — Earnings Surprise Analysis for Amazon (AMZN)

**السؤال:** احسب الـ median لعائد يومين (2-day %) بعد أيام الـ positive earnings surprise.

In [ ]:
ticker_obj = yf.Ticker('AMZN')
earnings_dates_df = ticker_obj.get_earnings_dates(limit=30)
print(earnings_dates_df.shape)
earnings_dates_df.head(10)


In [ ]:
amzn_hist = yf.download('AMZN', period='max', interval='1d', progress=False)
amzn_hist = amzn_hist.sort_index()
close_col_amzn = 'Close' if 'Close' in amzn_hist.columns else amzn_hist.columns[0]
amzn_close = amzn_hist[close_col_amzn].dropna()

print(f"AMZN price history: {amzn_close.index.min().date()} to {amzn_close.index.max().date()}")


In [ ]:
def two_day_return_after(event_date: pd.Timestamp, price_series: pd.Series):
    """
    Day1 = أول يوم تداول عند/بعد تاريخ الحدث (ممكن يكون هو يوم الإعلان أو أقرب يوم بعده)
    Day3 = بعد يومين تداول من Day1 (positional indexing)
    Return = Close_Day3 / Close_Day1 - 1
    """
    idx = price_series.index
    valid_dates = idx[idx >= event_date]
    if len(valid_dates) == 0:
        return np.nan
    pos = idx.get_loc(valid_dates[0])
    if pos + 2 >= len(idx):
        return np.nan
    day1_price = price_series.iloc[pos]
    day3_price = price_series.iloc[pos + 2]
    if pd.isna(day1_price) or pd.isna(day3_price) or day1_price == 0:
        return np.nan
    return day3_price / day1_price - 1


earnings_analysis = earnings_dates_df.copy()
earnings_analysis.index = pd.to_datetime(earnings_analysis.index).tz_localize(None)

earnings_analysis['2d_return'] = [
    two_day_return_after(d, amzn_close) for d in earnings_analysis.index
]

# عمود الـ Surprise % ممكن يكون اسمه مختلف شوية حسب نسخة yfinance
surprise_col_candidates = [c for c in earnings_analysis.columns if 'surprise' in c.lower()]
print("Surprise column(s) found:", surprise_col_candidates)
surprise_col = surprise_col_candidates[0] if surprise_col_candidates else None

earnings_analysis = earnings_analysis.dropna(subset=['2d_return'])
earnings_analysis.head(10)


In [ ]:
if surprise_col is not None:
    earnings_analysis[surprise_col] = pd.to_numeric(earnings_analysis[surprise_col], errors='coerce')

    positive_surprises = earnings_analysis[earnings_analysis[surprise_col] > 0]

    median_2d_return_positive = positive_surprises['2d_return'].median()
    print(f"Number of positive-surprise earnings events: {len(positive_surprises)}")
    print(f">>> Median 2-day return after POSITIVE earnings surprises = {median_2d_return_positive:.2%}")

    corr_matrix = earnings_analysis[[surprise_col, '2d_return']].corr()
    print("\nCorrelation matrix:")
    display(corr_matrix)

    correlation = corr_matrix.loc[surprise_col, '2d_return']
    print(f"\n>>> Correlation between surprise magnitude and 2-day return = {correlation:.3f}")
else:
    print("[WARN] Surprise % column not found in this yfinance version — تأكد من التحديث: pip install -U yfinance")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
if surprise_col is not None:
    ax.scatter(earnings_analysis[surprise_col], earnings_analysis['2d_return'], color='steelblue', edgecolor='black')
    ax.axhline(0, color='gray', linestyle=':')
    ax.axvline(0, color='gray', linestyle=':')
    ax.set_xlabel('Earnings Surprise (%)')
    ax.set_ylabel('2-Day Return')
    ax.set_title('AMZN: Earnings Surprise vs. 2-Day Stock Return')
    plt.tight_layout()
    plt.show()


## ملخص الإجابات النهائية

In [ ]:
print("Q1) أعلى سنة إضافات لمؤشر S&P 500 (من 2020) =", best_year,
      "| عدد الأسهم في المؤشر من أكتر من 20 سنة =", len(stocks_over_20y))
print("Q2) عدد المؤشرات (من 10) اللي أداؤها أفضل من S&P 500 YTD حتى 21 أغسطس 2026 =", len(better_than_us))
print("Q3) Median drawdown للتصحيحات الملحوظة (S&P 500, 1950-present) =", f"{median_drawdown:.1f}%",
      "| median duration =", f"{median_duration:.0f} يوم")
if surprise_col is not None:
    print("Q4) Median عائد يومين بعد positive earnings surprise (AMZN) =", f"{median_2d_return_positive:.2%}",
          "| correlation مع حجم المفاجأة =", f"{correlation:.3f}")


## Question 5 — [استكشافي، اختياري] فكرة مشروع التخرج (Capstone)

> مثال للإجابة (عدّلها حسب اهتمامك الفعلي):

أنا مهتم ببناء نموذج تنبؤ قصير المدى (30-60 يوم) لأسهم الشركات التقنية متوسطة القيمة السوقية في السوق الأمريكي،
باستخدام مزيج من:
- مؤشرات فنية (RSI, MACD, Bollinger Bands)
- بيانات الاكتتابات الحديثة (IPO momentum) زي اللي حللناها في Module 2
- عوامل ماكرو (أسعار الفائدة، مؤشر VIX)

الهدف: نموذج تصنيف (classification) يتنبأ هل السهم هيحقق عائد أعلى من السوق (S&P 500) خلال فترة زمنية محددة ولا لأ.

## Question 6 — [استكشافي، اختياري] استكشاف مقاييس (metrics) جديدة

> مثال للإجابة (عدّلها حسب فكرة مشروعك من سؤال 5):

مقاييس إضافية ممكن تفيد المشروع:
1. **VIX Index (^VIX)** — مؤشر "الخوف" في السوق، بيتحمّل عبر `yfinance` بنفس طريقة تحميل المؤشرات فوق، ومفيد كـ feature يوضح حالة تقلب السوق العامة وقت اتخاذ قرار الشراء.
2. **US 10-Year Treasury Yield (^TNX)** — يعكس تكلفة الفرصة البديلة للاستثمار في الأسهم، ومفيد لحساب مقاييس زي Sharpe Ratio بدقة أكبر بدل استخدام رقم ثابت.
3. **Insider Trading Data** (من مصادر زي SEC EDGAR) — نشاط تداول المديرين التنفيذيين ممكن يكون إشارة مبكرة على ثقتهم في أداء السهم المستقبلي.